# Mini-TP 2 — Actividad entregable (Sesión 2)

**Expón los metadatos de tu modelo por GraphQL y compáralo con REST.**
Individual · entrega esta semana. Completa las celdas marcadas con `# TODO`.

## Consigna
1. Define un **esquema GraphQL** (Strawberry) con un tipo `Model` (`name`, `version`, `metrics`).
2. Una **query** que devuelva las métricas/experimentos de tu modelo (de MLflow local o simuladas).
3. Pruébalo desde **GraphiQL** y desde un **cliente Python**.
4. **Compara** la misma lectura contra tu endpoint **REST** de la Sesión 1 (llamadas y datos) y anota la diferencia.

**Se evalúa:** que corra de punta a punta; que el esquema tipe entrada y salida; que la query pida solo lo necesario; y la reflexión REST vs GraphQL.
**Opcional (+):** que el resolver lea el **linaje** desde Neo4j (ver `graphql_neo4j_lineage.ipynb`).

## 0. Requisitos
```bash
uv pip install strawberry-graphql fastapi "uvicorn[standard]" requests mlflow neo4j
```
(`mlflow` para leer los metadatos del punto 1; `neo4j` para el linaje del punto 5.)

In [1]:
# !uv pip install strawberry-graphql fastapi "uvicorn[standard]" requests mlflow neo4j

## 1. Los metadatos de TU modelo

Como estamos continuando el mismo proyecto/modelo trabajado en MLOps1, leemos los metadatos del **Model Registry de MLflow** (`http://localhost:5000`): tomamos la versión en `Production` (o la última registrada) y sus métricas del run que la generó.

Si MLflow no está disponible (notebook offline), se usa un **diccionario de respaldo** con los últimos valores conocidos, para que el resto del notebook corra igual. `MI_MODELO`, en cualquiera de los dos casos, siempre queda con la misma forma: `name`, `version`, `metrics` (+ `run_id` y `stage` si vino de MLflow).

In [2]:
import requests

MLFLOW_TRACKING_URI = "http://localhost:5000"
MODEL_NAME = "xgb_best"
METRICAS = ("mae", "rmse", "mape", "r2")

# Diccionario de respaldo (últimos valores conocidos), si MLflow no está disponible.
MI_MODELO_FALLBACK = {
    "name": MODEL_NAME,
    "version": 1,
    "run_id": "a26e6742b3494869a7592a3ac62dc0c6",
    "stage": "Production",
    "metrics": {
        "mae": 72742.55404226085,
        "rmse": 134160.91132937567,
        "mape": 0.1570270434560993,
        "r2": 0.936149537563324,
    },
    "source": "fallback",
}


def _mlflow_vivo(uri: str, timeout: float = 2.0) -> bool:
    """Ping rápido al endpoint de salud de MLflow; evita esperar el backoff largo del cliente."""
    try:
        return requests.get(f"{uri}/health", timeout=timeout).ok
    except requests.RequestException:
        return False


def _leer_de_mlflow(name: str) -> dict:
    """Lee name/version/metrics del Model Registry de MLflow.

    - Elige la versión en stage 'Production'; si no hay, la de mayor número.
    - Trae las métricas del run que generó esa versión.
    """
    import mlflow
    from mlflow.tracking import MlflowClient

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    client = MlflowClient()

    versiones = client.search_model_versions(f"name = '{name}'")
    if not versiones:
        raise RuntimeError(f"'{name}' no tiene versiones registradas en MLflow")

    en_prod = [v for v in versiones if v.current_stage == "Production"]
    mv = en_prod[0] if en_prod else max(versiones, key=lambda v: int(v.version))

    run = client.get_run(mv.run_id)
    m = run.data.metrics
    faltantes = [k for k in METRICAS if k not in m]
    if faltantes:
        raise RuntimeError(f"El run {mv.run_id} no tiene las métricas {faltantes}")

    return {
        "name": name,
        "version": int(mv.version),
        "run_id": mv.run_id,
        "stage": mv.current_stage,
        "metrics": {k: float(m[k]) for k in METRICAS},
        "source": "mlflow",
    }


if _mlflow_vivo(MLFLOW_TRACKING_URI):
    try:
        MI_MODELO = _leer_de_mlflow(MODEL_NAME)
        print(f"Metadatos leídos de MLflow ({MLFLOW_TRACKING_URI}).")
    except Exception as e:
        MI_MODELO = MI_MODELO_FALLBACK
        print(f"MLflow respondió pero falló la lectura ({type(e).__name__}: {e}).")
        print("Uso el diccionario de respaldo.")
else:
    MI_MODELO = MI_MODELO_FALLBACK
    print(f"MLflow no responde en {MLFLOW_TRACKING_URI}. Uso el diccionario de respaldo.")

MI_MODELO


Metadatos leídos de MLflow (http://localhost:5000).


{'name': 'xgb_best',
 'version': 1,
 'run_id': 'a26e6742b3494869a7592a3ac62dc0c6',
 'stage': 'Production',
 'metrics': {'mae': 72742.55404226085,
  'rmse': 134160.91132937567,
  'mape': 0.1570270434560993,
  'r2': 0.936149537563324},
 'source': 'mlflow'}

## 2. El esquema GraphQL

Completa el tipo `Model` y el resolver que devuelve sus métricas.

In [3]:
import strawberry
from typing import Optional
from fastapi import FastAPI
from strawberry.fastapi import GraphQLRouter

@strawberry.type
class Metrics:
    mae: float
    rmse: float
    mape: float
    r2: float

@strawberry.type
class Model:
    name: str
    version: int

    @strawberry.field
    def metrics(self) -> Metrics:
        m = MI_MODELO["metrics"]
        # TODO: si lees de MLflow, trae las métricas reales aquí
        return Metrics(mae=m["mae"], rmse=m["rmse"], mape=m["mape"], r2=m["r2"])

@strawberry.type
class Query:
    @strawberry.field
    def model(self) -> Model:
        return Model(name=MI_MODELO["name"], version=MI_MODELO["version"])

schema = strawberry.Schema(query=Query)
app = FastAPI(title="Mini-TP 2 - metadatos por GraphQL")
app.include_router(GraphQLRouter(schema), prefix="/graphql")
print("Esquema listo")

Esquema listo


## 3. Levantar la API (en segundo plano) y consultar

In [4]:
import threading, time, uvicorn
def _run():
    uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8010, log_level="warning")).run()
threading.Thread(target=_run, daemon=True).start(); time.sleep(2)

import requests
query = "{ model { name version metrics { mae rmse mape r2 } } }"
r = requests.post("http://127.0.0.1:8010/graphql", json={"query": query})
print(r.status_code, r.json())

200 {'data': {'model': {'name': 'xgb_best', 'version': 1, 'metrics': {'mae': 72742.55404226085, 'rmse': 134160.91132937567, 'mape': 0.1570270434560993, 'r2': 0.936149537563324}}}}


## 4. Comparación con REST 

Sirve el mismo modelo por REST (puedes reusar tu API de la Sesión 1) y cuenta cuántas llamadas y cuántos bytes necesitas para armar la misma vista. Compara con GraphQL.

> Referencia: `rest_vs_graphql.ipynb` hace exactamente esta comparación con código.

In [5]:
# TODO: tu comparación REST vs GraphQL (llamadas y bytes) y una breve conclusión escrita.
from fastapi import FastAPI, HTTPException, Security, APIRouter
from fastapi.security.api_key import APIKeyHeader
from pydantic import BaseModel, Field

app = FastAPI(title="ML API - REST")

# ---------- REST ----------

# --- SEGURIDAD ---
api_key_header = APIKeyHeader(name="X-API-KEY", auto_error=False)
CLIENTES_AUTORIZADOS = {"token-secreto-123": "Cliente_Alpha"}

def validar_token(api_key: str = Security(api_key_header)):
    if api_key not in CLIENTES_AUTORIZADOS:
        raise HTTPException(status_code=403, detail="Token inválido.")
    return CLIENTES_AUTORIZADOS[api_key]

# --- ESQUEMAS ---
class Metrics_REST(BaseModel):
    mae: float
    rmse: float
    mape: float
    r2: float

class Caracteristicas(BaseModel):
    name: str = Field()
    version: int = Field()

    def metrics(self) -> Metrics_REST:
        m = MI_MODELO["metrics"]
        return Metrics(mae=m["mae"], rmse=m["rmse"], mape=m["mape"], r2=m["r2"])

# --- REGISTRO DE MODELOS (Simulado) ---
def _by_name(mname: str):
    return MI_MODELO if MI_MODELO["name"] == mname else None

class ModeloV1():
    def predecir(self, f1: float, f2: float) -> int:
        return 1 if (f1 * 0.5 + f2) > 10 else 0  # Lógica antigua

    def modelos(self):
        # Objeto (casi) completo, SIN metrics -> para el dashboard hará falta otra llamada por modelo
        return [{k: v for k, v in MI_MODELO.items() if k != "metrics"} ]

    def metricas(self, mname: str):
        m = _by_name(mname)
        if not m:
            raise HTTPException(status_code=404, detail="modelo no encontrado")
        return m["metrics"]

models_list = {"v1": ModeloV1()}

# --- ENRUTAMIENTO VERSIONADO ---
router_v1 = APIRouter(prefix="/v1", tags=["Modelo V1"])

@router_v1.get("/rest/models")
def rest_models():
    # Objeto (casi) completo, SIN metrics -> para el dashboard hará falta otra llamada por modelo
    return models_list["v1"].modelos()

@router_v1.get("/rest/models/{mname}/metrics")
def rest_metrics(mname: str):
    return models_list["v1"].metricas(mname)

app.include_router(router_v1)

import threading, time, uvicorn
def _run():
    uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8020, log_level="warning")).run()
threading.Thread(target=_run, daemon=True).start(); time.sleep(2)

import requests
query = "{}"
r = requests.get("http://127.0.0.1:8020/v1/rest/models")
print(r.status_code, r.json())
r = requests.get("http://127.0.0.1:8020/v1/rest/models/xgb_best/metrics")
print(r.status_code, r.json())


200 [{'name': 'xgb_best', 'version': 1, 'run_id': 'a26e6742b3494869a7592a3ac62dc0c6', 'stage': 'Production', 'source': 'mlflow'}]
200 {'mae': 72742.55404226085, 'rmse': 134160.91132937567, 'mape': 0.1570270434560993, 'r2': 0.936149537563324}


### 4.1 Comparación REST vs GraphQL (números + conclusión)

Vista objetivo: **el nombre y las métricas de mi modelo** (`mae`, `rmse`, `mape`, `r2`).

- **REST:** `GET /v1/rest/models` no trae las métricas → hace falta **una llamada más por modelo** a `GET /v1/rest/models/{name}/metrics` (patrón *N+1*). Además `/v1/rest/models` podría devolver campos que no necesito (*over-fetching*).
- **GraphQL:** una sola query pide `name` + `metrics { mae rmse mape r2 }` y devuelve *exactamente* eso.

Medimos **número de llamadas** y **bytes transferidos** (`len(response.content)`).

In [7]:
import requests

REST_BASE = "http://127.0.0.1:8020"
GQL_URL   = "http://127.0.0.1:8010/graphql"

# ---------- REST: armar la vista (name + metrics) ----------
llamadas_rest = 0
bytes_rest = 0

r = requests.get(f"{REST_BASE}/v1/rest/models"); llamadas_rest += 1; bytes_rest += len(r.content)
dashboard_rest = []
for m in r.json():
    rm = requests.get(f"{REST_BASE}/v1/rest/models/{m['name']}/metrics")
    llamadas_rest += 1; bytes_rest += len(rm.content)
    dashboard_rest.append((m["name"], rm.json()))

print("Vista REST:", dashboard_rest)
print(f"REST    -> {llamadas_rest} llamadas, {bytes_rest} bytes transferidos")

# ---------- GraphQL: una sola query, exactamente lo pedido ----------
query = """
query {
  model {
    name
    metrics { mae rmse mape r2 }
  }
}
"""
rg = requests.post(GQL_URL, json={"query": query})
llamadas_gql = 1
bytes_gql = len(rg.content)
d = rg.json()["data"]["model"]
dashboard_gql = [(d["name"], d["metrics"])]

print("Vista GraphQL:", dashboard_gql)
print(f"GraphQL -> {llamadas_gql} llamada, {bytes_gql} bytes transferidos")

# ---------- La comparación, en números ----------
print()
print(f"{'':12}{'llamadas':>10}{'bytes':>10}")
print(f"{'REST':12}{llamadas_rest:>10}{bytes_rest:>10}")
print(f"{'GraphQL':12}{llamadas_gql:>10}{bytes_gql:>10}")
print()
_rel = bytes_rest / max(bytes_gql, 1)
print(f"GraphQL usó {llamadas_rest / llamadas_gql:.0f}x menos llamadas y "
      f"~{_rel:.1f}x {'menos' if _rel >= 1 else 'más'} datos para el MISMO resultado.")


Vista REST: [('xgb_best', {'mae': 72742.55404226085, 'rmse': 134160.91132937567, 'mape': 0.1570270434560993, 'r2': 0.936149537563324})]
REST    -> 2 llamadas, 216 bytes transferidos
Vista GraphQL: [('xgb_best', {'mae': 72742.55404226085, 'rmse': 134160.91132937567, 'mape': 0.1570270434560993, 'r2': 0.936149537563324})]
GraphQL -> 1 llamada, 161 bytes transferidos

              llamadas     bytes
REST                 2       216
GraphQL              1       161

GraphQL usó 2x menos llamadas y ~1.3x menos datos para el MISMO resultado.


In [8]:
# Detalle del over-fetching: qué trajo /v1/rest/models que NO necesitábamos para la vista
lista = requests.get(f"{REST_BASE}/v1/rest/models").json()
campos_traidos = set().union(*[set(m.keys()) for m in lista])
campos_necesarios = {"name"}  # para la vista solo queríamos el nombre + sus métricas
print("Campos que devolvió /v1/rest/models:", sorted(campos_traidos))
print("Campos realmente necesarios       :", sorted(campos_necesarios))
print("Over-fetching (campos de más)      :", sorted(campos_traidos - campos_necesarios) or "ninguno")
print()
print("El endpoint /v1/rest/models expone el objeto (casi) completo del modelo: además de")
print("'name' devuelve 'version', 'run_id', 'stage', 'source'... que para armar la vista")
print("'nombre + métricas' no hacían falta. Con GraphQL el cliente pide solo 'name'.")


Campos que devolvió /v1/rest/models: ['name', 'run_id', 'source', 'stage', 'version']
Campos realmente necesarios       : ['name']
Over-fetching (campos de más)      : ['run_id', 'source', 'stage', 'version']

El endpoint /v1/rest/models expone el objeto (casi) completo del modelo: además de
'name' devuelve 'version', 'run_id', 'stage', 'source'... que para armar la vista
'nombre + métricas' no hacían falta. Con GraphQL el cliente pide solo 'name'.


### 4.2 Conclusión — REST vs GraphQL para leer los metadatos de mi modelo

Números medidos en este notebook (1 modelo, `xgb_best`):

| | llamadas | bytes |
|---|---|---|
| **REST** (`:8020/v1/...`) | **2** (1 lista + 1 métricas) | 216 |
| **GraphQL** (`:8010/graphql`) | **1** | 161 |

| Aspecto | REST | GraphQL |
|---|---|---|
| Llamadas para la vista (name + metrics) | `n + 1` → **2** con 1 modelo | **1** |
| Datos de más (*over-fetching*) | Sí: `/v1/rest/models` trae `name`+`version`+`run_id`+`stage`+`source`  | **Ninguno**: la respuesta trae solo los campos de la query |
| *Under-fetching* / N+1 | Sí: las métricas viven en otro endpoint → una llamada por modelo | No: `metrics` se resuelve en la misma query |
| Forma de la respuesta | Fija por endpoint | La define el cliente (`mae` solo, o los 4 campos) |
| Caching HTTP | Simple y maduro (GET cacheable por URL) | Más complejo (POST, misma URL para todo) |
| Tipado entrada/salida | Pydantic por endpoint | Esquema Strawberry (`Model`, `Metrics`) |

**Qué diferencia noté:** para leer "nombre + métricas" de un modelo, REST necesita **2 llamadas** (patrón *N+1*: primero la lista, después las métricas de cada modelo por separado); GraphQL lo resuelve en **1 sola llamada** pidiendo exactamente los 4 campos de métricas.

En **bytes**, acá REST transfirió un poco **más** (216 vs 133). Y esta relación se hace mas evidente a favor de GraphQL en cuanto crece el escenario: cada modelo extra en la vista es, en REST, otra llamada de métricas (más *round-trips* y más *overhead* de HTTP por request), mientras que en GraphQL sigue siendo **1 llamada**; y cada campo relacionado que quiera agregar es una línea más en la query en vez de un endpoint nuevo. La contracara: REST mantiene el *caching* HTTP simple y un endpoint por recurso.



## 5. (Opcional +) Linaje desde Neo4j

Agregamos al tipo `Model` un campo **`lineage`** cuyo resolver **ejecuta Cypher contra Neo4j**. Así, en **una sola query GraphQL** el cliente pide metadatos + métricas + linaje del modelo — el linaje sale de un grafo, no de un diccionario en Python.

**Grafo de linaje real de `xgb_best`** — reconstruido del proyecto de MLOps1 en el que se usó un pipeline de Airflow `train_xgb_pipeline` y un modelo registrado en MLflow (`http://localhost:5000`, experimento `car_price_xgb`):

```
(Dataset: raw-data/car_details_v3.csv)
      │ DERIVES  (task preprocess del DAG)
      ▼
(Feature: car_features_v3)   ── car_age, km_driven, mileage_value, engine_value,
      │ DERIVES                  max_power_value, torque_nm, seats, owner_num,
      ▼                          is_manual, is_individual + one-hot top-10 marcas
(Experiment: car_price_xgb)
      │ DERIVES
      ▼
(Run: xgb_best_training)   ── run_id a26e6742…  (XGBRegressor, hiperparámetros de Optuna)
      │ DERIVES
      ▼
(Model: xgb_best@1)   ── stage=Production
      │ DERIVES
      ▼
(Deployment: mlops-api)   ── contenedor mlops1_final-api, sirve el modelo en :8000
```

Fuente de cada nodo:

| Nodo | Valor real | De dónde sale |
|---|---|---|
| `Dataset` | `raw-data/car_details_v3.csv` | DAG: `RAW_BUCKET="raw-data"`, `OBJECT_KEY="car_details_v3.csv"` (bucket MinIO) |
| `Feature` | `car_features_v3` | task `preprocess` del DAG + `common/constants.py` (`NUMERIC_FEATURES`, `BINARY_FEATURES`, top-10 marcas) |
| `Experiment` | `car_price_xgb` | `mlflow.set_experiment("car_price_xgb")` en el DAG |
| `Run` | `xgb_best_training` / `a26e6742b3494869a7592a3ac62dc0c6` | MLflow Tracking API |
| `Model` | `xgb_best`, versión 1, `Production` | MLflow Model Registry |
| `Deployment` | `mlops-api` (`mlops1_final-api`, puerto 8000) | contenedor que sirve el modelo |

**Requisitos:**
```bash
uv pip install neo4j
```
```bash
# Neo4j en Docker (terminal aparte, dejar corriendo):
docker run --name neo4j-tp -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/testpass neo4j:latest
```
> Se probó contra un contenedor Neo4j activo con auth `neo4j/testpass`. Hay que ajustar debajo las constantes `NEO4J_*` si se usa otra clave/puerto. El *seed* usa `MERGE` acotado a los nodos de linaje: **no borra** otros grafos de la misma base.

### 5.1 Conexión a Neo4j y siembra del grafo de linaje

El driver se crea una sola vez. El *seed* usa `MERGE` (idempotente, no toca otros nodos) para crear la cadena real `Dataset → Feature → Experiment → Run → Model → Deployment` con sus propiedades, unida por relaciones `DERIVES`.

In [10]:
import os
from neo4j import GraphDatabase

NEO4J_URI      = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "testpass")

_driver = None
def get_driver():
    """Crea el driver de Neo4j una sola vez (no por request)."""
    global _driver
    if _driver is None:
        _driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    return _driver

# --- Linaje REAL de xgb_best (pipeline Airflow train_xgb_pipeline + MLflow car_price_xgb) ---
MODEL_TAG = f'{MI_MODELO["name"]}@{MI_MODELO["version"]}'   # -> "xgb_best@1"
MLFLOW_RUN_ID = MI_MODELO.get("run_id", MI_MODELO["run_id"]) # -> "run_id" de MLflow o fallback

# NOTA de Cypher: para crear/asegurar un nodo se usa MERGE (crea si no existe) o
# CREATE (siempre nuevo).
# MERGE identifica el nodo por las props del patrón; SET agrega/actualiza el resto.
SEED = [
    # --- nodos con sus propiedades reales ---
    ("MERGE (d:Dataset {name:'raw-data/car_details_v3.csv'}) "
     "SET d.bucket='raw-data', d.object_key='car_details_v3.csv', d.store='MinIO'"),
    ("MERGE (f:Feature {name:'car_features_v3'}) "
     "SET f.target='log1p(selling_price)', "
     "f.numeric='car_age,km_driven,mileage_value,engine_value,max_power_value,torque_nm,seats,owner_num', "
     "f.binary='is_manual,is_individual', f.categorical='one-hot top-10 brands'"),
    ("MERGE (e:Experiment {name:'car_price_xgb'}) SET e.tracking_uri='http://localhost:5000'"),
    ("MERGE (r:Run {name:'xgb_best_training'}) "
     "SET r.run_id=$run_id, r.estimator='XGBRegressor', r.orchestrator='airflow', "
     "r.hpo='optuna', r.n_estimators=400, r.max_depth=6, r.learning_rate=0.029282661153638836"),
    ("MERGE (m:Model {name:$model_tag}) "
     "SET m.registry_name='xgb_best', m.version=1, m.stage='Production', "
     "m.mae=$mae, m.rmse=$rmse, m.mape=$mape, m.r2=$r2"),
    ("MERGE (dep:Deployment {name:'mlops-api'}) "
     "SET dep.container='mlops1_final-api', dep.port=8000, dep.type='REST'"),
    # --- relaciones DERIVES: Dataset -> Feature -> Experiment -> Run -> Model -> Deployment ---
    ("MATCH (d:Dataset {name:'raw-data/car_details_v3.csv'}), (f:Feature {name:'car_features_v3'}) "
     "MERGE (d)-[:DERIVES]->(f)"),
    ("MATCH (f:Feature {name:'car_features_v3'}), (e:Experiment {name:'car_price_xgb'}) "
     "MERGE (f)-[:DERIVES]->(e)"),
    ("MATCH (e:Experiment {name:'car_price_xgb'}), (r:Run {name:'xgb_best_training'}) "
     "MERGE (e)-[:DERIVES]->(r)"),
    ("MATCH (r:Run {name:'xgb_best_training'}), (m:Model {name:$model_tag}) "
     "MERGE (r)-[:DERIVES]->(m)"),
    ("MATCH (m:Model {name:$model_tag}), (dep:Deployment {name:'mlops-api'}) "
     "MERGE (m)-[:DERIVES]->(dep)"),
]

_METRICS = MI_MODELO["metrics"]
_PARAMS = dict(model_tag=MODEL_TAG, run_id=MLFLOW_RUN_ID,
               mae=_METRICS["mae"], rmse=_METRICS["rmse"],
               mape=_METRICS["mape"], r2=_METRICS["r2"])

LINEAGE_LABELS = ["Dataset", "Feature", "Experiment", "Run", "Model", "Deployment"]
_LABEL_FILTER = " OR ".join(f"n:{lbl}" for lbl in LINEAGE_LABELS)

def seed_lineage():
    get_driver().verify_connectivity()
    with get_driver().session() as ses:
        for stmt in SEED:
            ses.run(stmt, **_PARAMS)
    print(f"Grafo de linaje sembrado para {MODEL_TAG!r}.")

def clean_lineage():
    """Borra SOLO los nodos de linaje de este TP (no toca otros grafos)."""
    with get_driver().session() as ses:
        ses.run(f"MATCH (n) WHERE {_LABEL_FILTER} DETACH DELETE n")
    print("Nodos de linaje eliminados.")

seed_lineage()

# Verificación: qué nodos quedaron en el grafo de linaje
with get_driver().session() as ses:
    filas = ses.run(
        f"MATCH (n) WHERE {_LABEL_FILTER} "
        "RETURN head(labels(n)) AS kind, n.name AS name ORDER BY kind, name"
    )
    print("Nodos de linaje en Neo4j:")
    for r in filas:
        print(f"  ({r['kind']}) {r['name']}")


Grafo de linaje sembrado para 'xgb_best@1'.
Nodos de linaje en Neo4j:
  (Dataset) raw-data/car_details_v3.csv
  (Deployment) mlops-api
  (Experiment) car_price_xgb
  (Feature) car_features_v3
  (Model) xgb_best@1
  (Run) xgb_best_training


### 5.2 Esquema GraphQL con el resolver de linaje

Nuevo tipo `Artifact` (`name` + `kind` = el label del nodo) y un campo `lineage` en `Model` cuyo resolver corre Cypher. `[:DERIVES*1..]` recorre la cadena `Dataset → Feature → Experiment → Run → Model` a **cualquier profundidad** — en SQL serían varios JOINs encadenados.

In [11]:
import strawberry
from typing import List
from fastapi import FastAPI
from strawberry.fastapi import GraphQLRouter

@strawberry.type
class MetricsL:
    mae: float
    rmse: float
    mape: float
    r2: float

@strawberry.type
class Artifact:
    name: str
    kind: str  # Dataset | Feature | Experiment | Run | Model | Deployment

@strawberry.type
class ModelL:
    name: str
    version: int

    @strawberry.field
    def metrics(self) -> MetricsL:
        m = MI_MODELO["metrics"]
        return MetricsL(mae=m["mae"], rmse=m["rmse"], mape=m["mape"], r2=m["r2"])

    @strawberry.field
    def lineage(self) -> List[Artifact]:
        """Artefactos aguas arriba de este modelo (su linaje), desde Neo4j.

        [:DERIVES*1..] recorre la cadena Dataset -> Feature -> Experiment -> Run -> Model
        a cualquier profundidad. En SQL serían varios JOINs encadenados.
        """
        cypher = """
        MATCH (a)-[:DERIVES*1..]->(m:Model {name: $tag})
        RETURN DISTINCT a.name AS name, head(labels(a)) AS kind
        """
        tag = f"{self.name}@{self.version}"
        with get_driver().session() as ses:
            rows = ses.run(cypher, tag=tag)
            return [Artifact(name=r["name"], kind=r["kind"]) for r in rows]

@strawberry.type
class QueryL:
    @strawberry.field
    def model(self) -> ModelL:
        return ModelL(name=MI_MODELO["name"], version=MI_MODELO["version"])

schema_l = strawberry.Schema(query=QueryL)
app_l = FastAPI(title="Mini-TP 2 - metadatos + linaje por GraphQL")
app_l.include_router(GraphQLRouter(schema_l), prefix="/graphql")
print("Esquema con linaje listo")


Esquema con linaje listo


### 5.3 Levantar la API y consultar linaje + métodos en UNA query

Levantamos esta API en el puerto **8030** (en segundo plano). Después pedimos, en **una sola llamada GraphQL**: `name`, `version`, `metrics` (de Python) y `lineage` (de Neo4j).

In [12]:
import threading, time, socket, uvicorn, requests, json

def _port_libre(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex((host, port)) != 0  # != 0 -> nadie escucha, puerto libre

def _run_l():
    uvicorn.Server(uvicorn.Config(app_l, host="127.0.0.1", port=8030, log_level="warning")).run()

# Solo levantamos el server si el puerto está libre (evita [Errno 10048] al re-ejecutar).
if _port_libre("127.0.0.1", 8030):
    threading.Thread(target=_run_l, daemon=True).start()
    time.sleep(2)
else:
    print("Puerto 8030 ya en uso: reuso el server que está corriendo.")

GQL_L = "http://127.0.0.1:8030/graphql"
query = """
query {
  model {
    name
    version
    metrics { mae rmse mape r2 }
    lineage { name kind }
  }
}
"""
r = requests.post(GQL_L, json={"query": query})
print(r.status_code)
print(json.dumps(r.json(), indent=2))


200
{
  "data": {
    "model": {
      "name": "xgb_best",
      "version": 1,
      "metrics": {
        "mae": 72742.55404226085,
        "rmse": 134160.91132937567,
        "mape": 0.1570270434560993,
        "r2": 0.936149537563324
      },
      "lineage": [
        {
          "name": "xgb_best_training",
          "kind": "Run"
        },
        {
          "name": "car_price_xgb",
          "kind": "Experiment"
        },
        {
          "name": "car_features_v3",
          "kind": "Feature"
        },
        {
          "name": "raw-data/car_details_v3.csv",
          "kind": "Dataset"
        }
      ]
    }
  }
}


### 5.4 Con linaje, ¿cómo queda REST vs GraphQL?

Ahora la vista es **name + version + metrics + linaje**. En REST el linaje sería *otro* endpoint (`GET /v1/rest/models/{name}/lineage`), así que sumamos una llamada más al patrón N+1. En GraphQL sigue siendo **1 sola query** — solo se agregó `lineage { name kind }`.

In [13]:
# --- REST: simulamos en REST el endpoint de linaje que haría falta, usando GraphQL ---
def rest_lineage(mname: str):
    cypher = """
    MATCH (a)-[:DERIVES*1..]->(m:Model {name: $tag})
    RETURN DISTINCT a.name AS name, head(labels(a)) AS kind
    """
    tag = f'{mname}@{MI_MODELO["version"]}'
    with get_driver().session() as ses:
        return [dict(r) for r in ses.run(cypher, tag=tag)]

# ---------- REST: armar la vista name + version + metrics + linaje ----------
llamadas_rest = 0
bytes_rest = 0

r = requests.get(f"{REST_BASE}/v1/rest/models"); llamadas_rest += 1; bytes_rest += len(r.content)
vista_rest = []
for m in r.json():
    rm = requests.get(f"{REST_BASE}/v1/rest/models/{m['name']}/metrics")
    llamadas_rest += 1; bytes_rest += len(rm.content)
    lin = rest_lineage(m["name"])                       # <-- endpoint de linaje (otra llamada)
    payload_lin = json.dumps(lin).encode()
    llamadas_rest += 1; bytes_rest += len(payload_lin)
    vista_rest.append({"name": m["name"], "version": m["version"],
                       "metrics": rm.json(), "lineage": lin})

print("Vista REST:", json.dumps(vista_rest, indent=2))
print(f"REST    -> {llamadas_rest} llamadas, {bytes_rest} bytes")

# ---------- GraphQL: la misma vista, 1 query ----------
rg = requests.post(GQL_L, json={"query": query})
llamadas_gql = 1
bytes_gql = len(rg.content)
print(f"GraphQL -> {llamadas_gql} llamada, {bytes_gql} bytes")

print()
print(f"{'':12}{'llamadas':>10}{'bytes':>10}")
print(f"{'REST':12}{llamadas_rest:>10}{bytes_rest:>10}")
print(f"{'GraphQL':12}{llamadas_gql:>10}{bytes_gql:>10}")
print()
print(f"Para la vista CON linaje: GraphQL usó {llamadas_rest}x menos llamadas y "
      f"~{bytes_rest/max(bytes_gql,1):.1f}x {'menos' if bytes_rest >= bytes_gql else 'más'} datos.")


Vista REST: [
  {
    "name": "xgb_best",
    "version": 1,
    "metrics": {
      "mae": 72742.55404226085,
      "rmse": 134160.91132937567,
      "mape": 0.1570270434560993,
      "r2": 0.936149537563324
    },
    "lineage": [
      {
        "name": "xgb_best_training",
        "kind": "Run"
      },
      {
        "name": "car_price_xgb",
        "kind": "Experiment"
      },
      {
        "name": "car_features_v3",
        "kind": "Feature"
      },
      {
        "name": "raw-data/car_details_v3.csv",
        "kind": "Dataset"
      }
    ]
  }
]
REST    -> 3 llamadas, 419 bytes
GraphQL -> 1 llamada, 391 bytes

              llamadas     bytes
REST                 3       419
GraphQL              1       391

Para la vista CON linaje: GraphQL usó 3x menos llamadas y ~1.1x menos datos.


### 5.5 Conclusión del punto 5

El linaje de `xgb_best@1` es un dato **relacional** que vive en **otra fuente** (un grafo Neo4j) y se reconstruyó de artefactos reales del proyecto de MLOps1:

```
raw-data/car_details_v3.csv  →  car_features_v3  →  car_price_xgb  →  xgb_best_training  →  xgb_best@1
     (Dataset, MinIO)            (Feature)          (Experiment)        (Run MLflow)         (Model)
```

La query GraphQL del punto 5.3 devuelve esos **4 artefactos aguas arriba** (`Dataset`, `Feature`, `Experiment`, `Run`) — no incluye el `Deployment` porque está *aguas abajo* del modelo (es consumidor, no origen).

| | llamadas | de dónde sale cada campo |
|---|---|---|
| **REST** | **3+** (lista + métricas + linaje), y crece con cada modelo o cada tramo de linaje que expongas | endpoints distintos, el cliente los orquesta |
| **GraphQL** | **1** | 1 query; `metrics` lo resuelve Python y `lineage` lo resuelve Cypher recorriendo `[:DERIVES*1..]`, transparente para el cliente |

- **`[:DERIVES*1..]`** recorre la cadena entera de un salto (5 niveles: `Dataset → Feature → Experiment → Run → Model`). En una base relacional serían 4 JOINs encadenados; en REST, un endpoint a medida por cada forma de la vista.
- El cliente pide la **forma exacta**: `lineage { kind }` para un resumen, `lineage { name kind }` para el detalle — la misma API, sin endpoints nuevos.
- **GraphQL** es la puerta flexible y tipada; **Neo4j** es el motor que resuelve las relaciones. Es el patrón de *metadata & lineage* de una plataforma de MLOps: quién produjo este modelo, con qué datos, en qué experimento.



### Limpieza (opcional)

Para no dejar los nodos de linaje en el grafo, correr en una celda nueva:

```python
clean_lineage()
```

No lo dejamos como celda ejecutable para que el notebook entregado conserve la evidencia del punto 5 (el grafo sembrado y la query con `lineage` poblado).

---
### Qué entregar
* Este notebook corriendo de punta a punta.
* La query GraphQL y su salida.
* La comparación REST vs GraphQL (números + conclusión).
* En el README de tu repo, una línea sobre qué diferencia notaste.

    README: la diferencia en cantidad de llamadas es notable a favor de GraphQL, bajando significativamente a medida que agregamos modelos y/o versiones de los mismos. Algo similar sucede con los bytes transferidos de información, que son menos que con las llamadas REST